# 🏭 EEIO Carbon Emissions Simulator

Enter a company's **sales revenue** and **cost structure**, and this notebook automatically
calculates **Scope 1 · 2 · 3 carbon emissions** based on Korea's 2023 KR-EEIO
(Environmentally Extended Input-Output) data.

Scope 3 here refers to **supply-chain emissions** derived from the input-output table.
It therefore differs from the **GHG Protocol**, which also covers categories such as
**transport, business travel, and disposal** beyond purchased raw materials.

> **Required file**: an `.xlsx` file starting with `O_2023_Simulator` in the **same folder** as this notebook
> **How to run**: edit only the `STEP 1` ~ `STEP 4` cells below -> **Run All Cells**

> ⚙️ **This notebook is self-contained.** The loading/calculation/export code from
> `eeio_simulator.py` is copied directly into the setup cell below, so this notebook computes
> everything itself -- it does not `import eeio_simulator`.

---

All code and data for the KR-EEIO carbon-emissions analysis pipeline are available on the
[Data Science Team/kr_eeio Gitlab](https://bidas-gitlab.boknet.intra/2620316/kr_eeio/-/tree/main/) page.


## ⚙️ Initialization *(no changes needed)*

In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import glob
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter

# ──────────────────────────────────────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────────────────────────────────────
VA_NAMES = ["Compensation of employees", "Net operating surplus", "Consumption of fixed capital"]
TAX_NAME = "Other Taxes on production less subsidies / Taxes on production and products less subsidies"
TOL = 1e-6

FD_NAMES = {
    "9111": "Private final consumption expenditure (Households and NPISH)", "9112": "Government final consumption expenditure",
    "9121": "Private gross fixed capital formation", "9122": "Government gross fixed capital formation",
    "9131": "Changes in inventories", "9132": "Acquisitions less disposals of valuables", "9140": "Exports",
}

# Industry code -> name (normally auto-read from the source file; left empty here in advance)
_INDUSTRY_NAMES_FALLBACK: dict[str, str] = {}


# ──────────────────────────────────────────────────────────────────────────────
# Exception
# ──────────────────────────────────────────────────────────────────────────────
class UnprocessableError(Exception):
    """raised when the input parameters cannot be used to build the analysis table."""


# ══════════════════════════════════════════════════════════════════════════════
# 1. Source file detection & loading
# ══════════════════════════════════════════════════════════════════════════════

def find_simulator_file(search_dir: str | Path | None = None) -> Path:
    """Auto-detects a 2023_Simulator*.xlsx file in the same folder."""
    dirs = []
    if search_dir:
        dirs.append(Path(search_dir))

    # If running in an IPython (Jupyter) environment, also add the notebook location
    try:
        import IPython
        ip = IPython.get_ipython()
        if ip:
            for attr in ("__vsc_ipynb_file__", "_file_"):
                nb_file = getattr(ip, attr, None)
                if nb_file:
                    dirs.append(Path(nb_file).parent)
                    break
    except Exception:
        pass

    dirs.append(Path.cwd())

    patterns = ["*Simulator*.xlsx"]
    for d in dirs:
        for pat in patterns:
            hits = sorted(d.glob(pat))
            hits = [p for p in hits if p.parent.name != "output"]
            if hits:
                return hits[0]

    raise FileNotFoundError(
        "A simulator template .xlsx file starting with '2023_Simulator...' must be placed "
        "in the same folder as this notebook (one of: all costs known / partial costs known / sales revenue only)."
    )


# ──────────────────────────────────────────────────────────────────────────────
# Internal: sheet structure parsing
# ──────────────────────────────────────────────────────────────────────────────

def _find_data_start_row(ws, max_scan=20):
    for r in range(1, max_scan + 1):
        if ws.cell(row=r, column=1).value == "A":
            return r
    raise ValueError("Could not find the sector-classification start row")


def _find_data_start_col(ws, header_row, max_scan=20):
    for c in range(1, max_scan + 1):
        if ws.cell(row=header_row, column=c).value == "A":
            return c
    raise ValueError("Could not find the sector-classification start column")


def _parse_io_structure(ws, has_value_added=True, has_ghg=False):
    data_start_row = _find_data_start_row(ws)
    header_code_row = data_start_row - 2
    header_name_row = data_start_row - 1

    row_order, row_index = [], {}
    r = data_start_row
    while True:
        code = ws.cell(row=r, column=1).value
        if code is None:
            raise ValueError(f"{r}Row: sector the code is empty")
        if code == "9590":
            inter_total_row = r
            break
        row_order.append(code)
        row_index[code] = r
        r += 1

    va_rows, va_total_row, total_input_row, ghg_row = {}, None, None, None
    if has_value_added:
        r = inter_total_row + 1
        while True:
            code = ws.cell(row=r, column=1).value
            name = ws.cell(row=r, column=2).value
            if code == "9690":
                va_total_row = r
                r += 1
                if ws.cell(row=r, column=1).value == "9790":
                    total_input_row = r
                break
            va_rows[name] = r
            r += 1
        if has_ghg and ws.cell(row=total_input_row + 1, column=1).value == "GHG":
            ghg_row = total_input_row + 1

    data_start_col = _find_data_start_col(ws, header_code_row)
    col_order, col_index = [], {}
    c = data_start_col
    while True:
        code = ws.cell(row=header_code_row, column=c).value
        if code is None:
            raise ValueError(f"{get_column_letter(c)}Column: sector the code is empty")
        if code == "9090":
            inter_demand_col = c
            break
        col_order.append(code)
        col_index[code] = c
        c += 1

    fd_cols, fd_total_col, total_demand_col = {}, None, None
    c = inter_demand_col + 1
    while True:
        code = ws.cell(row=header_code_row, column=c).value
        if code == "9190":
            fd_total_col = c
        elif code == "9290":
            total_demand_col = c
            break
        else:
            fd_cols[code] = c
        c += 1

    assert row_order == col_order, "The row/column sector-classification orders do not match"

    return dict(
        data_start_row=data_start_row, header_code_row=header_code_row,
        header_name_row=header_name_row, row_order=row_order, row_index=row_index,
        inter_total_row=inter_total_row, va_rows=va_rows, va_total_row=va_total_row,
        total_input_row=total_input_row, ghg_row=ghg_row,
        data_start_col=data_start_col, col_order=col_order, col_index=col_index,
        inter_demand_col=inter_demand_col, fd_cols=fd_cols,
        fd_total_col=fd_total_col, total_demand_col=total_demand_col,
    )


def _parse_coef_structure(ws):
    data_start_row = _find_data_start_row(ws)
    header_code_row = data_start_row - 2
    row_order, row_index = [], {}
    r = data_start_row
    while True:
        code = ws.cell(row=r, column=1).value
        if code == "9590":
            inter_total_row = r
            break
        row_order.append(code)
        row_index[code] = r
        r += 1
    va_rows = {}
    r = inter_total_row + 1
    while True:
        code = ws.cell(row=r, column=1).value
        name = ws.cell(row=r, column=2).value
        if code == "9690" or code is None:
            break
        va_rows[name] = r
        r += 1
    data_start_col = _find_data_start_col(ws, header_code_row)
    col_order, col_index = [], {}
    c = data_start_col
    while True:
        code = ws.cell(row=header_code_row, column=c).value
        if code is None or code == "9090":
            break
        col_order.append(code)
        col_index[code] = c
        c += 1
    return dict(row_index=row_index, col_index=col_index, va_rows=va_rows)


def _get_industry_names(ws, struct):
    return {code: ws.cell(row=r, column=2).value for code, r in struct["row_index"].items()}


def _extract_arrays(ws, struct):
    codes = struct["row_order"]
    n = len(codes)
    fd_codes = sorted(struct["fd_cols"])
    n_fd = len(fd_codes)

    mat = np.zeros((n, n))
    fd = np.zeros((n, n_fd))
    for i, ci in enumerate(codes):
        ri = struct["row_index"][ci]
        for j, cj in enumerate(codes):
            v = ws.cell(row=ri, column=struct["col_index"][cj]).value
            mat[i, j] = v if v is not None else 0.0
        for k, fc in enumerate(fd_codes):
            v = ws.cell(row=ri, column=struct["fd_cols"][fc]).value
            fd[i, k] = v if v is not None else 0.0

    out = dict(codes=codes, fd_codes=fd_codes, mat=mat, fd=fd)

    if struct.get("va_rows"):
        va = np.zeros((3, n))
        tax = np.zeros(n)
        for j, cj in enumerate(codes):
            col = struct["col_index"][cj]
            for v_i, name in enumerate(VA_NAMES):
                vv = ws.cell(row=struct["va_rows"][name], column=col).value
                va[v_i, j] = vv if vv is not None else 0.0
            tv = ws.cell(row=struct["va_rows"][TAX_NAME], column=col).value
            tax[j] = tv if tv is not None else 0.0
        out["va"] = va
        out["tax"] = tax

        total_output = np.zeros(n)
        for j, cj in enumerate(codes):
            tv = ws.cell(row=struct["total_input_row"], column=struct["col_index"][cj]).value
            total_output[j] = tv if tv is not None else 0.0
        out["total_output"] = total_output

    if struct.get("ghg_row"):
        ghg = np.zeros(n)
        for j, cj in enumerate(codes):
            gv = ws.cell(row=struct["ghg_row"], column=struct["col_index"][cj]).value
            ghg[j] = gv if gv is not None else 0.0
        out["ghg"] = ghg

    return out


def _extract_coef_arrays(ws, coef_struct, codes):
    n = len(codes)
    mat = np.zeros((n, n))
    va = np.zeros((3, n))
    for j, cj in enumerate(codes):
        cidx = coef_struct["col_index"][cj]
        for i, ci in enumerate(codes):
            v = ws.cell(row=coef_struct["row_index"][ci], column=cidx).value
            mat[i, j] = v if v is not None else 0.0
        for v_i, name in enumerate(VA_NAMES):
            v = ws.cell(row=coef_struct["va_rows"][name], column=cidx).value
            va[v_i, j] = v if v is not None else 0.0
    return mat, va


# ══════════════════════════════════════════════════════════════════════════════
# 2. Public API: data loading
# ══════════════════════════════════════════════════════════════════════════════

def load_eeio_data(src_path: str | Path | None = None) -> dict:
    """
    Loads the raw EEIO data from the simulator template xlsx file.

    Parameters
    ----------
    src_path : str | Path | None
        Path to the xlsx file. If None, auto-detected from the current folder.

    Returns
    -------
    dict  {
        "src_path"       : Path,
        "output_dir"     : Path,
        "codes"          : list[str],          # 33 sector codes
        "industry_names" : dict[str, str],
        "A_data"         : dict,               # Total Transaction table arrays
        "D_data"         : dict,               # Domestic Transaction table arrays
        "coef_mat"       : np.ndarray,         # total input coefficients
    }
    """
    src = Path(src_path) if src_path else find_simulator_file()
    output_dir = src.parent / "output"
    output_dir.mkdir(exist_ok=True)

    print(f"📂 Source file: {src.name}")

    wb = openpyxl.load_workbook(src, data_only=True)

    ws_A = wb["A_Total_Trans_Producer_Orig"]
    A_struct = _parse_io_structure(ws_A, has_value_added=True, has_ghg=True)
    industry_names = _get_industry_names(ws_A, A_struct)
    A_data = _extract_arrays(ws_A, A_struct)

    ws_D = wb["Domestic_Transaction_Orig"]
    D_struct = _parse_io_structure(ws_D, has_value_added=False)
    D_data = _extract_arrays(ws_D, D_struct)

    ws_coef = wb["Total_Input_Coeff_A"]
    coef_struct = _parse_coef_structure(ws_coef)
    coef_mat, _ = _extract_coef_arrays(ws_coef, coef_struct, A_data["codes"])

    print(f"✅ Data loaded  (sector {len(A_data['codes'])})")

    return dict(
        src_path=src,
        output_dir=output_dir,
        codes=A_data["codes"],
        industry_names=industry_names,
        A_data=A_data,
        D_data=D_data,
        coef_mat=coef_mat,
    )


# ══════════════════════════════════════════════════════════════════════════════
# 3. Internal: cost allocate / row·column insertion / Ad·Lf·M calculate
# ══════════════════════════════════════════════════════════════════════════════

def _allocate_costs(known_intermediate, known_va, codes, revenue, coef_mat, host_idx):
    n = len(codes)
    L = np.zeros(n)
    is_known = np.zeros(n, dtype=bool)
    for i, c in enumerate(codes):
        if c in known_intermediate:
            L[i] = known_intermediate[c]
            is_known[i] = True

    L_va = np.zeros(3)
    for v_i, name in enumerate(VA_NAMES):
        if name in known_va:
            L_va[v_i] = known_va[name]

    known_total = L[is_known].sum() + L_va.sum()
    remaining = revenue - known_total

    if remaining < -TOL:
        raise UnprocessableError(
            f"⛔ not possible to process: the sum of known costs({known_total:,.1f}) Sales revenue({revenue:,.1f}) exceeds."
        )

    K_est = coef_mat[:, host_idx] * revenue
    denom = K_est[~is_known].sum()
    if remaining > TOL:
        if denom <= TOL:
            raise UnprocessableError(
                "⛔ not possible to process: to allocate the remaining budget to there is no unknown intermediate-input item."
            )
        L[~is_known] = K_est[~is_known] / denom * remaining

    return L, L_va


def _four_point_balance(mat, h, company_idx):
    """Pure 4-Point cross-balancing (removing negative cells)."""
    N = mat.shape[0]
    for i in range(N):
        if mat[i, h] < -1e-6:
            deficit = abs(mat[i, h]); mat[i, h] = 0.0
            tr = mat[i, :].copy(); tr[h] = -np.inf; tr[company_idx] = -np.inf
            mc = np.argmax(tr)
            tc = mat[:, h].copy(); tc[i] = -np.inf; tc[company_idx] = -np.inf
            mr = np.argmax(tc)
            mat[i, mc] -= deficit; mat[mr, h] -= deficit; mat[mr, mc] += deficit
    for j in range(N):
        if mat[h, j] < -1e-6:
            deficit = abs(mat[h, j]); mat[h, j] = 0.0
            tr = mat[h, :].copy(); tr[j] = -np.inf; tr[company_idx] = -np.inf
            mc = np.argmax(tr)
            tc = mat[:, j].copy(); tc[h] = -np.inf; tc[company_idx] = -np.inf
            mr = np.argmax(tc)
            mat[h, mc] -= deficit; mat[mr, j] -= deficit; mat[mr, mc] += deficit


def _build_A_new(A_data, host_idx, revenue, L, L_va):
    codes = A_data["codes"]; n = len(codes)
    mat, fd, va, tax = A_data["mat"], A_data["fd"], A_data["va"], A_data["tax"]
    n_fd = fd.shape[1]; h = host_idx; ci = h + 1; N = n + 1

    def m(i): return i if i <= h else i + 1

    T_h = mat[h, :].sum() + fd[h, :].sum()
    scale = revenue / T_h if T_h > 0 else 0.0

    nm = np.zeros((N, N)); nf = np.zeros((N, n_fd))
    nv = np.zeros((3, N)); nt = np.zeros(N)

    for i in range(n):
        if i != h:
            for j in range(n):
                if j != h:
                    nm[m(i), m(j)] = mat[i, j]
            nm[m(i), h] = mat[i, h] - L[i]
            nf[m(i), :] = fd[i, :]
        nm[m(i), ci] = L[i]

    nm[ci, ci] = 0.0; nm[h, ci] = L[h]
    for j in range(n):
        if j != h:
            nm[ci, m(j)] = mat[h, j] * scale
    nm[ci, h] = mat[h, h] * scale
    for k in range(n_fd):
        nf[ci, k] = fd[h, k] * scale

    for j in range(n):
        if j != h:
            nm[h, m(j)] = mat[h, j] - nm[ci, m(j)]
    nm[h, h] = mat[h, h] - nm[h, ci] - nm[ci, h]
    for k in range(n_fd):
        nf[h, k] = fd[h, k] - nf[ci, k]

    for j in range(n):
        nj = m(j)
        if j != h:
            nv[:, nj] = va[:, j]; nt[nj] = tax[j]
        else:
            nv[:, ci] = L_va; nv[:, h] = va[:, j] - L_va
            nt[ci] = 0.0; nt[h] = tax[j]

    _four_point_balance(nm, h, ci)

    for v in range(3):
        if nv[v, h] < -1e-6:
            deficit = abs(nv[v, h]); nv[v, h] = 0.0
            tmp = nv[:, h].copy(); tmp[v] = -np.inf
            mv = np.argmax(tmp); nv[mv, h] -= deficit

    total_out = nm.sum(axis=0) + nv.sum(axis=0) + nt
    new_codes = codes[:ci] + ["Company under analysis"] + codes[ci:]
    return dict(codes=new_codes, mat=nm, fd=nf, va=nv, tax=nt,
                total_output=total_out, host_idx=h, company_idx=ci)


def _build_D_new(D_data, A_data, A_new, host_idx):
    codes = D_data["codes"]; n = len(codes)
    dmat, dfd = D_data["mat"], D_data["fd"]
    amat = A_data["mat"]; h = host_idx; ci = h + 1; N = n + 1
    full = A_new["mat"]

    def m(i): return i if i <= h else i + 1
    def ratio(a, b): return a / b if abs(b) > 1e-9 else 0.0

    nm = np.zeros((N, N)); nf = np.zeros((N, dfd.shape[1]))

    for i in range(n):
        if i != h:
            for j in range(n):
                if j != h:
                    nm[m(i), m(j)] = dmat[i, j]
            nm[m(i), ci] = ratio(dmat[i, h], amat[i, h]) * full[m(i), ci]
            nm[m(i), h] = dmat[i, h] - nm[m(i), ci]
            nf[m(i), :] = dfd[i, :]

    nm[ci, ci] = 0.0; nm[h, ci] = ratio(dmat[h, h], amat[h, h]) * full[h, ci]
    for j in range(N):
        if j != ci:
            nm[ci, j] = full[ci, j]
    for k in range(dfd.shape[1]):
        nf[ci, k] = A_new["fd"][ci, k]

    for j in range(n):
        if j != h:
            nm[h, m(j)] = dmat[h, j] - nm[ci, m(j)]
    nm[h, h] = dmat[h, h] - nm[h, ci] - nm[ci, h]
    for k in range(dfd.shape[1]):
        nf[h, k] = dfd[h, k] - nf[ci, k]

    _four_point_balance(nm, h, ci)

    new_codes = codes[:ci] + ["Company under analysis"] + codes[ci:]
    return dict(codes=new_codes, mat=nm, fd=nf)


def _build_ghg_new(A_data, host_idx, revenue):
    codes = A_data["codes"]; ghg = A_data["ghg"]; total = A_data["total_output"]
    n = len(codes); h = host_idx; ci = h + 1; N = n + 1

    def m(i): return i if i <= h else i + 1

    ng = np.zeros(N)
    for j in range(n):
        nj = m(j)
        if j != h:
            ng[nj] = ghg[j]
        else:
            cg = (ghg[j] / total[j] * revenue) if total[j] > 1e-9 else 0.0
            ng[ci] = cg; ng[h] = max(0.0, ghg[j] - cg)
    return ng


def _compute_ad_lf_m(A_new, D_new, new_ghg):
    total_output = A_new["total_output"]; N = len(total_output)
    with np.errstate(divide="ignore", invalid="ignore"):
        Ad = np.where(total_output[None, :] > 1e-9,
                      D_new["mat"] / total_output[None, :], 0.0)
        ghg_coef = np.where(total_output > 1e-9, new_ghg / total_output, 0.0)
    Lf = np.linalg.inv(np.eye(N) - Ad)
    M = ghg_coef[:, None] * Lf
    return Ad, Lf, M, ghg_coef, M.sum(axis=0)


# ══════════════════════════════════════════════════════════════════════════════
# 4. Public API: simulation run
# ══════════════════════════════════════════════════════════════════════════════

def run_simulation(data: dict, params: dict) -> dict:
    """
    Run the EEIO carbon-emission simulation.

    Parameters
    ----------
    data : dict returned by load_eeio_data()
    params : {
        "company_name" : str,          # company name (used in the result file name)
        "company_code" : str,          # the sector code it belongs to (e.g. "C05")
        "sales"        : float,        # sales revenue (KRW million)
        "cost_ratios"  : dict[str, float],   # cost shares (%, 0 if unknown)
        "va_ratios"    : dict[str, float],   # value-added shares (%, 0 if unknown)
    }

    Returns
    -------
    dict  {
        "company_name", "company_code", "industry_name",
        "sales", "case_label",
        "scope1", "scope2", "scope3", "total_emission",
        "scope_df"        : pd.DataFrame,   # Scope summary
        "industry_emit_df": pd.DataFrame,   # per-sector emissions contribution
        "A_new", "D_new", "Ad", "Lf", "M", "new_ghg",
        "L", "L_va", "codes_new",
    }
    """
    codes       = data["codes"]
    names       = data["industry_names"]
    A_data      = data["A_data"]
    D_data      = data["D_data"]
    coef_mat    = data["coef_mat"]

    company_name = params["company_name"]
    company_code = params["company_code"]
    sales        = params["sales"]
    cost_ratios  = params.get("cost_ratios", {c: 0 for c in codes})
    va_ratios    = params.get("va_ratios", {n: 0 for n in VA_NAMES})

    # ── verify ──
    def _is_num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)

    if not _is_num(sales) or sales <= 0:
        raise UnprocessableError(f"⛔ Sales revenue 0must be a number greater than. (input value: {sales!r})")
    if company_code not in codes:
        raise UnprocessableError(f"⛔ sector code '{company_code}' could not be found. possible Code: {codes}")
    for label, d in [("cost", cost_ratios), ("Value added", va_ratios)]:
        for k, v in d.items():
            if not _is_num(v):
                raise UnprocessableError(f"⛔ {label} '{k}' value is not a number: {v!r}")
            if v < 0:
                raise UnprocessableError(f"⛔ {label} '{k}' value is negative: {v}")
    unknown = [c for c in cost_ratios if c not in codes]
    if unknown:
        raise UnprocessableError(f"⛔ Unknown cost code: {unknown}")

    cost_pct = sum(cost_ratios.values())
    va_pct   = sum(va_ratios.values())
    total_pct = cost_pct + va_pct
    if total_pct > 100.0 + TOL:
        raise UnprocessableError(
            f"⛔ cost({cost_pct}%) + Value added({va_pct}%) = {total_pct}% > 100%  "
            "— Sales revenuegreater than costs cannot be allocated."
        )

    if total_pct >= 100.0 - TOL:
        case_label = "Case3 (all costs known)"
    elif total_pct > 0:
        case_label = "Case2 (partial costs known)"
    else:
        case_label = "Case1 (sales revenue only known)"

    known_inter = {c: p / 100 * sales for c, p in cost_ratios.items() if p > 0}
    known_va    = {n: p / 100 * sales for n, p in va_ratios.items() if p > 0}

    host_idx = codes.index(company_code)
    L, L_va  = _allocate_costs(known_inter, known_va, codes, sales, coef_mat, host_idx)

    col_in = L.sum() + L_va.sum()
    if abs(col_in - sales) > 1e-3:
        raise UnprocessableError(f"⛔ cost sum of the allocated({col_in:,.3f}) ≠ Sales revenue({sales:,.3f})")

    # ── row/column insertion ──
    A_new   = _build_A_new(A_data, host_idx, sales, L, L_va)
    D_new   = _build_D_new(D_data, A_data, A_new, host_idx)
    new_ghg = _build_ghg_new(A_data, host_idx, sales)

    ci = A_new["company_idx"]
    Ad, Lf, M, ghg_coef, total_induced = _compute_ad_lf_m(A_new, D_new, new_ghg)

    elec_orig = codes.index("D")
    elec_new  = elec_orig if elec_orig <= host_idx else elec_orig + 1

    scope1 = float(ghg_coef[ci] * sales)
    scope2 = float(M[elec_new, ci] * sales)
    total_emission = float(total_induced[ci] * sales)
    scope3 = total_emission - scope1 - scope2

    # ── Scope Summary DataFrame ──
    scope_df = pd.DataFrame({
        "item": ["Scope1 (direct emissions)", "Scope2 (indirect emissions from electricity, etc.)", "Scope3 (other indirect emissions)"],
        "emissions (tCO2eq.)": [scope1, scope2, scope3],
        "share": [
            f"{scope1 / total_emission * 100:.1f}%" if total_emission else "–",
            f"{scope2 / total_emission * 100:.1f}%" if total_emission else "–",
            f"{scope3 / total_emission * 100:.1f}%" if total_emission else "–",
        ],
    })

    # ── by sector emissions contribution DataFrame ──
    new_codes = A_new["codes"]
    m_col = M[:, ci] * sales          # by sector induceemissions
    industry_emit = []
    for idx, c in enumerate(new_codes):
        nm = "Company under analysis" if c == "Company under analysis" else names.get(c, c)
        industry_emit.append({
            "Code": c if c != "Company under analysis" else "–",
            "Sector name": nm,
            "emissions (tCO2eq.)": float(m_col[idx]),
            "share": f"{m_col[idx] / total_emission * 100:.1f}%" if total_emission else "–",
        })
    industry_emit_df = (
        pd.DataFrame(industry_emit)
        .sort_values("emissions (tCO2eq.)", ascending=False)
        .reset_index(drop=True)
    )

    return dict(
        company_name=company_name,
        company_code=company_code,
        industry_name=names.get(company_code, company_code),
        sales=sales,
        case_label=case_label,
        scope1=scope1, scope2=scope2, scope3=scope3,
        total_emission=total_emission,
        scope_df=scope_df,
        industry_emit_df=industry_emit_df,
        A_new=A_new, D_new=D_new,
        Ad=Ad, Lf=Lf, M=M,
        new_ghg=new_ghg,
        L=L, L_va=L_va,
        codes_new=new_codes,
        industry_names=names,
        fd_codes=A_data["fd_codes"],
    )


# ══════════════════════════════════════════════════════════════════════════════
# 5. Public API: Result visualization
# ══════════════════════════════════════════════════════════════════════════════

def _html_table(df, col_formats=None, row_colors=None, title=None, center_cols=None,
                total_row=None):
    col_formats = col_formats or {}
    th_base = (
        "background:#2c3e50;color:white;padding:9px 16px;"
        "font-size:13px;white-space:nowrap;"
    )
    td_base = "padding:8px 16px;font-size:13px;border-bottom:1px solid #e0e0e0;"
    tf_base = (
        "padding:8px 16px;font-size:13px;font-weight:bold;"
        "border-top:2px solid #2c3e50;background:#f0f4f8;"
    )

    def _align(col):
        if center_cols is None or col in center_cols:
            return "text-align:center;"
        return "text-align:left;"

    rows_html = []
    for _, row in df.iterrows():
        extra = row_colors(row) if row_colors else ""
        cells = []
        for col in df.columns:
            val = row[col]
            if col in col_formats and isinstance(val, (int, float)):
                val = col_formats[col].format(val)
            cells.append(f"<td style='{td_base}{_align(col)}{extra}'>{val}</td>")
        rows_html.append(f"<tr>{''.join(cells)}</tr>")

    if total_row is not None:
        total_cells = []
        for col in df.columns:
            val = total_row.get(col, "")
            if col in col_formats and isinstance(val, (int, float)):
                val = col_formats[col].format(val)
            total_cells.append(f"<td style='{tf_base}{_align(col)}'>{val}</td>")
        rows_html.append(f"<tr>{''.join(total_cells)}</tr>")

    headers = "".join(
        f"<th style='{th_base}{_align(c)}'>{c}</th>" for c in df.columns
    )
    table = (
        f"<table style='border-collapse:collapse;width:100%;margin-bottom:4px'>"
        f"<thead><tr>{headers}</tr></thead>"
        f"<tbody>{''.join(rows_html)}</tbody>"
        f"</table>"
    )
    if title:
        table = f"<h3 style='margin-bottom:6px'>{title}</h3>" + table
    return table


def display_results(result: dict) -> None:
    """
    Displays the result of run_simulation() as an HTML table in the Jupyter
    notebook.

    Parameters
    ----------
    result : dict returned by run_simulation()
    """
    from IPython.display import display, HTML

    # ── Company information ──
    info_df = pd.DataFrame({
        "item": ["Company name", "Industry", "Sales revenue (KRW million)", "Calculation mode"],
        "Content": [
            result["company_name"],
            f"{result['company_code']}  {result['industry_name']}",
            f"{result['sales']:,.0f}",
            result["case_label"],
        ],
    })
    display(HTML(_html_table(info_df, center_cols=["Content"], title="🏢 Company information")))

    # ── Scope Summary ──
    SCOPE_COLORS = {"Scope1": "#e63946", "Scope2": "#e07b39", "Scope3": "#457b9d"}

    def _scope_color(row):
        for k, c in SCOPE_COLORS.items():
            if k in str(row["item"]):
                return f"background:{c};color:white;font-weight:bold;"
        return ""

    display(HTML(_html_table(
        result["scope_df"],
        col_formats={"emissions (tCO2eq.)": "{:,.1f}"},
        row_colors=_scope_color,
        center_cols=["emissions (tCO2eq.)", "share"],
        title="🌍 carbon emissions result — Scope 1 · 2 · 3",
        total_row={
            "item": "Total (Scope 1+2+3)",
            "emissions (tCO2eq.)": result["total_emission"],
            "share": "100.0%",
        },
    )))

    # ── by sector emission contribution ──
    emit_df = result["industry_emit_df"].copy()
    emit_df.insert(0, "Rank", range(1, len(emit_df) + 1))

    def _emit_color(row):
        if row["Sector name"] == "Company under analysis":
            return "background:#e63946;color:white;font-weight:bold;"
        if row["Rank"] == 1:
            return "background:#ffd166;font-weight:bold;"
        if row["Rank"] <= 3:
            return "background:#fff3cd;"
        return ""

    display(HTML(_html_table(
        emit_df,
        col_formats={"emissions (tCO2eq.)": "{:,.1f}"},
        row_colors=_emit_color,
        center_cols=["Rank", "emissions (tCO2eq.)", "share"],
        title="🏭 by sector emission contribution (overall / Company under analysis direct emissions)",
        total_row={
            "Rank": "",
            "Code": "",
            "Sector name": "Total",
            "emissions (tCO2eq.)": result["total_emission"],
            "share": "100.0%",
        },
    )))


# ══════════════════════════════════════════════════════════════════════════════
# 6. Public API: Excel save
# ══════════════════════════════════════════════════════════════════════════════

def save_excel(result: dict, output_dir: str | Path) -> Path:
    """simulation the result Excel file saves."""
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    A_new      = result["A_new"]
    D_new      = result["D_new"]
    Ad         = result["Ad"]
    Lf         = result["Lf"]
    M          = result["M"]
    new_ghg    = result["new_ghg"]
    new_codes  = result["codes_new"]
    names      = result["industry_names"]
    fd_codes   = result["fd_codes"]

    def _display_names():
        return ["Company under analysis" if c == "Company under analysis" else names.get(c, "") for c in new_codes]

    new_names = _display_names()

    def _write_matrix(wb, title, codes, dnames, fd_codes, mat, fd,
                      va=None, tax=None, ghg=None):
        ws = wb.create_sheet(title)
        n = len(codes); n_fd = fd.shape[1]; fd_start = 3 + n
        ws.cell(row=5, column=1, value="Commodity")
        for j, (c, nm) in enumerate(zip(codes, dnames)):
            ws.cell(row=5, column=3 + j, value=c if c != "Company under analysis" else None)
            ws.cell(row=6, column=3 + j, value=nm)
        for k, fc in enumerate(fd_codes):
            ws.cell(row=5, column=fd_start + k, value=fc)
            ws.cell(row=6, column=fd_start + k, value=FD_NAMES.get(str(fc), str(fc)))
        for i, (c, nm) in enumerate(zip(codes, dnames)):
            r = 7 + i
            ws.cell(row=r, column=1, value=c if c != "Company under analysis" else None)
            ws.cell(row=r, column=2, value=nm)
            for j in range(n):
                ws.cell(row=r, column=3 + j, value=float(mat[i, j]))
            for k in range(n_fd):
                ws.cell(row=r, column=fd_start + k, value=float(fd[i, k]))
        if va is not None:
            cur = 7 + n
            for v_i, vname in enumerate(VA_NAMES + [TAX_NAME]):
                ws.cell(row=cur, column=2, value=vname)
                for j in range(n):
                    val = va[v_i, j] if v_i < 3 else tax[j]
                    ws.cell(row=cur, column=3 + j, value=float(val))
                cur += 1
            if ghg is not None:
                ws.cell(row=cur, column=1, value="GHG")
                ws.cell(row=cur, column=2, value="Direct GHG emissions (GHG)")
                for j in range(n):
                    ws.cell(row=cur, column=3 + j, value=float(ghg[j]))
        ws.column_dimensions["B"].width = 26
        return ws

    def _write_square(wb, title, codes, dnames, arr):
        ws = wb.create_sheet(title)
        n = len(codes)
        for j, (c, nm) in enumerate(zip(codes, dnames)):
            ws.cell(row=5, column=3 + j, value=c if c != "Company under analysis" else None)
            ws.cell(row=6, column=3 + j, value=nm)
        for i, (c, nm) in enumerate(zip(codes, dnames)):
            r = 7 + i
            ws.cell(row=r, column=1, value=c if c != "Company under analysis" else None)
            ws.cell(row=r, column=2, value=nm)
            for j in range(n):
                ws.cell(row=r, column=3 + j, value=float(arr[i, j]))
        ws.column_dimensions["B"].width = 26

    wb_out = openpyxl.Workbook()
    wb_out.remove(wb_out.active)

    # cases sheet
    ws_c = wb_out.create_sheet("cases")
    ws_c["A1"] = "Company information"; ws_c["A2"] = "Company name"; ws_c["B2"] = result["company_name"]
    ws_c["A3"] = "Industry"; ws_c["B3"] = f"{result['company_code']} ({result['industry_name']})"
    ws_c["A4"] = "Calculation mode"; ws_c["B4"] = result["case_label"]
    ws_c["A5"] = "Sales revenue(KRW million)"; ws_c["B5"] = result["sales"]
    ws_c["A7"] = "Cost allocation results"

    # header
    ws_c["A8"] = "Industry code"
    ws_c["B8"] = "Sector name"
    ws_c["C8"] = "Intermediate input (KRW million)"
    ws_c["D8"] = "share (%)"

    # L is indexed by the original codes (before the company row is inserted); same order as codes_new with the company excluded
    L_vec   = result["L"]      # np.ndarray, shape (n,)
    L_va    = result["L_va"]   # np.ndarray, shape (3,)
    orig_codes = [c for c in result["codes_new"] if c != "Company under analysis"]
    orig_names = result["industry_names"]
    sales_val  = result["sales"]

    row = 9
    for i, c in enumerate(orig_codes):
        nm  = orig_names.get(c, c)
        val = float(L_vec[i])
        pct = val / sales_val * 100 if sales_val > 0 else 0.0
        ws_c.cell(row=row, column=1, value=c)
        ws_c.cell(row=row, column=2, value=nm)
        ws_c.cell(row=row, column=3, value=round(val, 3))
        ws_c.cell(row=row, column=4, value=round(pct, 4))
        row += 1

    # Intermediate input subtotal
    total_inter = float(L_vec.sum())
    ws_c.cell(row=row, column=2, value="Intermediate input subtotal")
    ws_c.cell(row=row, column=3, value=round(total_inter, 3))
    ws_c.cell(row=row, column=4, value=round(total_inter / sales_val * 100, 4) if sales_val else 0.0)
    row += 1

    # Value added 3item
    for v_i, vname in enumerate(VA_NAMES):
        val = float(L_va[v_i])
        pct = val / sales_val * 100 if sales_val > 0 else 0.0
        ws_c.cell(row=row, column=2, value=vname)
        ws_c.cell(row=row, column=3, value=round(val, 3))
        ws_c.cell(row=row, column=4, value=round(pct, 4))
        row += 1

    # Value added subtotal
    total_va = float(L_va.sum())
    ws_c.cell(row=row, column=2, value="Value added subtotal")
    ws_c.cell(row=row, column=3, value=round(total_va, 3))
    ws_c.cell(row=row, column=4, value=round(total_va / sales_val * 100, 4) if sales_val else 0.0)
    row += 1

    # Total
    ws_c.cell(row=row, column=2, value="Total (Sales revenue)")
    ws_c.cell(row=row, column=3, value=round(total_inter + total_va, 3))
    ws_c.cell(row=row, column=4, value=round((total_inter + total_va) / sales_val * 100, 4) if sales_val else 0.0)

    ws_c.column_dimensions["A"].width = 10
    ws_c.column_dimensions["B"].width = 28
    ws_c.column_dimensions["C"].width = 22
    ws_c.column_dimensions["D"].width = 14

    # Summary sheet
    ws_s = wb_out.create_sheet("Summary")
    ws_s["A1"] = "Company name"; ws_s["B1"] = result["company_name"]
    ws_s["A2"] = "Industry"; ws_s["B2"] = f"{result['company_code']} ({result['industry_name']})"
    ws_s["A3"] = "Sales revenue(KRW million)"; ws_s["B3"] = result["sales"]
    ws_s["A4"] = "Calculation mode"; ws_s["B4"] = result["case_label"]
    ws_s["A6"] = "Scope1 (direct emissions, tCO2eq.)"; ws_s["B6"] = result["scope1"]
    ws_s["A7"] = "Scope2 (indirect emissions from electricity, tCO2eq.)"; ws_s["B7"] = result["scope2"]
    ws_s["A8"] = "Scope3 (other indirect emissions, tCO2eq.)"; ws_s["B8"] = result["scope3"]
    ws_s["A9"] = "Total carbon emissions (tCO2eq.)"; ws_s["B9"] = result["total_emission"]
    ws_s.column_dimensions["A"].width = 30

    # Transaction table / coefficient sheets
    _write_matrix(wb_out, "A_Total_Trans_Producer_New", new_codes, new_names, fd_codes,
                  A_new["mat"], A_new["fd"], A_new["va"], A_new["tax"], ghg=new_ghg)
    _write_matrix(wb_out, "Domestic_Transaction_New", new_codes, new_names, fd_codes,
                  D_new["mat"], D_new["fd"])
    _write_square(wb_out, "Ad", new_codes, new_names, Ad)
    _write_square(wb_out, "Lf", new_codes, new_names, Lf)
    _write_square(wb_out, "M", new_codes, new_names, M)

    safe = "".join(ch for ch in result["company_name"] if ch.isalnum() or ch in " _-").strip() or "Company"
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    out  = output_dir / f"{safe}_{result['company_code']}_CarbonEmissions_{ts}.xlsx"
    wb_out.save(out)
    print(f"💾 Result saved: {out}")
    return out


# ── run the loader directly ─────────────────────────────────────────────
DATA = load_eeio_data()


---
## 📝 STEP 1. Company name

In [ ]:
COMPANY_NAME = "BOK_Company"    # <- enter the company name (used in the result file name)


---
## 📝 STEP 2. Sector code

| Code | Sector | Code | Sector |
|------|--------|------|--------|
| `A` | Agricultural, forest, and fishery goods | `J` | Communications and broadcasting |
| `B` | Mined and quarried goods | `K` | Finance and insurance |
| `C01` | Food, beverages and tobacco products | `L` | Real estate services |
| `C02` | Textile and leather products | `M` | Professional, scientific, and technical services |
| `C03` | Wood and paper products, printing | `N` | Business support services |
| `C04` | Petroleum and coal products | `O` | Public administration, defense, and social security |
| `C05` | Chemical products | `P` | Education services |
| `C06` | Non-metallic mineral products | `Q` | Health and social care services |
| `C07` | Basic metal products | `R` | Art, sports, and leisure services |
| `C08` | Fabricated metal products | `S` | Other services |
| `C09` | Computing machinery, electronics, and optical instruments | `T` | Others |
| `C10` | Electrical equipment | `D` | Electricity, gas, and steam supply |
| `C11` | Machinery and equipment | `E` | Water supply, sewage, and waste treatment |
| `C12` | Transport equipment | `F` | Construction |
| `C13` | Other manufactured products | `G` | Wholesale/retail and commodity brokerage services |
| `C14` | Manufacturing services and repair of industrial equipment | `H` | Transportation services |
| `I` | Food services and accommodation | | |


In [ ]:
COMPANY_CODE = "C11"    # <- enter the matching code from the table above


---
## 📝 STEP 3. Sales revenue

> Unit: **KRW million** | e.g. KRW 1 billion -> `1_000` / KRW 1 trillion -> `1_000_000`


In [ ]:
SALES = 1000    # <- enter annual sales revenue in KRW million


---
## 📝 STEP 4a. Cost shares (COST_RATIOS)

Enter, for each sector, **what % of sales revenue** is spent purchasing from that sector.
Leave items you don't know as `0` and they will be **auto-estimated from the national average**.

> 💡 If you don't know any of the costs, leave everything at `0` -> Case 1 (sales revenue only known)
> 💡 If you know some of them, enter only the ones you know -> Case 2 (partial costs known)
> 💡 If you know all costs, COST_RATIOS + VA_RATIOS must sum to 100% -> Case 3


In [ ]:
COST_RATIOS = {
    "A":    0,    # Agricultural, forest, and fishery goods
    "B":    0,    # Mined and quarried goods
    "C01":  0,    # Food, beverages and tobacco products
    "C02":  0,    # Textile and leather products
    "C03":  5,    # Wood and paper products, printing
    "C04":  10,    # Petroleum and coal products
    "C05":  0,    # Chemical products
    "C06":  0,    # Non-metallic mineral products
    "C07":  0,    # Basic metal products
    "C08":  0,    # Fabricated metal products
    "C09":  0,    # Computing machinery, electronics, and optical instruments
    "C10":  0,    # Electrical equipment
    "C11":  0,   # Machinery and equipment
    "C12":  0,    # Transport equipment
    "C13":  0,    # Other manufactured products
    "C14":  10,    # Manufacturing services and repair of industrial equipment
    "D":    0,   # Electricity, gas, and steam supply
    "E":    0,    # Water supply, sewage, and waste treatment
    "F":    0,    # Construction
    "G":    0,    # Wholesale/retail and commodity brokerage services
    "H":    0,    # Transportation services
    "I":    0,    # Food services and accommodation
    "J":    0,    # Communications and broadcasting
    "K":    0,    # Finance and insurance
    "L":    0,    # Real estate services
    "M":    0,    # Professional, scientific, and technical services
    "N":    0,    # Business support services
    "O":    0,    # Public administration, defense, and social security
    "P":    0,    # Education services
    "Q":    0,    # Health and social care services
    "R":    0,    # Art, sports, and leisure services
    "S":    0,    # Other services
    "T":    0,    # Others
}


---
## 📝 STEP 4b. Value-added shares (VA_RATIOS)

Enter what % of sales revenue labor costs, operating profit, and depreciation each represent.
Leave as `0` if unknown and they will be auto-estimated from the national average.


In [ ]:
VA_RATIOS = {
    "Compensation of employees":            20,   # labor costs
    "Net operating surplus":                0,   # operating profit
    "Consumption of fixed capital":         0,   # depreciation
}


---
## 🚀 Run the calculation *(no changes needed)*

In [ ]:
RESULT = run_simulation(DATA, {
    "company_name": COMPANY_NAME,
    "company_code": COMPANY_CODE,
    "sales":        SALES,
    "cost_ratios":  COST_RATIOS,
    "va_ratios":    VA_RATIOS,
})


---
## 📊 View the results *(no changes needed)*

In [ ]:
display_results(RESULT)


---
## 💾 Save the results *(no changes needed)*

In [ ]:
from pathlib import Path

DATA["output_dir"] = Path("/tmp/eeio_output")
DATA["output_dir"].mkdir(parents=True, exist_ok=True)


In [ ]:
save_excel(RESULT, DATA["output_dir"])
